## **Import Libraries**

In [1]:
import math
import re
from collections import defaultdict

In [2]:
JOB_ROLES = [
    {
        "title": "Data Scientist",
        "skills": ["python", "machine learning", "sql", "statistics","data analysis", "tensorflow", "pandas", "numpy","deep learning", "data visualization"]
    },
    {
        "title": "Machine Learning Engineer",
        "skills": ["python", "machine learning", "tensorflow", "pytorch","deep learning", "algorithms", "data structures","docker", "kubernetes", "mlops"]
    },
    {
        "title": "Backend Developer",
        "skills": ["python", "java", "sql", "apis", "docker","nodejs", "databases", "git", "rest", "microservices"]
    },
    {
        "title": "Frontend Developer",
        "skills": ["javascript", "react", "css", "html", "typescript","nodejs", "git", "ui design", "responsive design", "web"]
    },
    {
        "title": "DevOps Engineer",
        "skills": ["aws", "docker", "kubernetes", "linux", "ci/cd","terraform", "git", "automation", "cloud", "bash"]
    },
    {
        "title": "Cloud Architect",
        "skills": ["aws", "azure", "cloud", "kubernetes", "terraform","networking", "security", "docker", "microservices", "devops"]
    },
    {
        "title": "Data Engineer",
        "skills": ["python", "sql", "spark", "hadoop", "etl","aws", "data pipelines", "databases", "kafka", "airflow"]
    },
    {
        "title": "Cybersecurity Analyst",
        "skills": ["networking", "security", "linux", "ethical hacking","firewalls", "penetration testing", "python", "cryptography","siem", "incident response"]
    },
    {
        "title": "Mobile Developer",
        "skills": ["java", "kotlin", "swift", "react", "flutter","apis", "git", "ui design", "android", "ios"]
    },
    {
        "title": "AI Research Scientist",
        "skills": ["python", "deep learning", "machine learning", "pytorch","mathematics", "statistics", "algorithms", "research","natural language processing", "computer vision"]
    },
    {
        "title": "Full Stack Developer",
        "skills": ["javascript", "python", "react", "nodejs", "sql","html", "css", "git", "apis", "docker"]
    },
    {
        "title": "Systems Administrator",
        "skills": ["linux", "networking", "bash", "automation", "security","aws", "virtualization", "git", "monitoring", "troubleshooting"]
    },
]

## **TF-IDF Vectorizer**

In [3]:
def compute_tfidf(roles):
    """
    Build TF-IDF vectors for all job roles
    Returns: (vocab list, list of TF-IDF dicts per role)
    """
    #Collect vocabulary (all unique skills across all roles)
    vocab = sorted(set(skill for role in roles for skill in role["skills"]))
    N = len(roles)
 
    # Document frequency: how many roles contain each skill
    df = defaultdict(int)
    for role in roles:
        for skill in set(role["skills"]):
            df[skill] += 1
 
    # IDF penalizes skills common to many roles
    idf = {skill: math.log(N / df[skill]) for skill in vocab}
 
    # TF-IDF per role
    tfidf_vectors = []
    for role in roles:
        tf = defaultdict(float)
        total_terms = len(role["skills"])
        for skill in role["skills"]:
            tf[skill] += 1 / total_terms   # TF = count / total terms
        vec = {skill: tf[skill] * idf[skill] for skill in vocab}
        tfidf_vectors.append(vec)
 
    return vocab, tfidf_vectors, idf

## **Build User Profile Vector**

In [4]:
def build_user_vector(user_skills, vocab, idf):
    """
    Convert user skill list into a TF-IDF weighted vector
    Skills not in vocab get zero weight (cold start handled gracefully)
    """
    total = len(user_skills)
    tf = defaultdict(float)
    matched = []
 
    for skill in user_skills:
        skill = skill.lower().strip()
        if skill in idf:
            tf[skill] += 1 / total
            matched.append(skill)
 
    vec = {skill: tf.get(skill, 0.0) * idf[skill] for skill in vocab}
    return vec, matched

## **Cosine Similarity**

In [5]:
def cosine_similarity(vec_a, vec_b):
    dot_product = sum(vec_a[k] * vec_b[k] for k in vec_a)
    mag_a = math.sqrt(sum(v ** 2 for v in vec_a.values()))
    mag_b = math.sqrt(sum(v ** 2 for v in vec_b.values()))
 
    if mag_a == 0 or mag_b == 0:
        return 0.0  
    return dot_product / (mag_a * mag_b)

## **Ranking PiPeline**

In [6]:
def recommend(user_skills, roles, tfidf_vectors, vocab, idf, top_n=3):
    """
    Full recommendation pipeline:
     Ingestion  — build user vector
     Scoring    — cosine similarity vs every role
     Sorting    — descending by score
     Filtering  — return Top-N only
    """
    #Ingestion
    user_vec, matched_skills = build_user_vector(user_skills, vocab, idf)
 
    #Cold start check
    if not matched_skills:
        return [], matched_skills, []
 
    #Scoring
    scored = []
    for i, role in enumerate(roles):
        score = cosine_similarity(user_vec, tfidf_vectors[i])
        scored.append((role["title"], score, role["skills"]))
 
    #sorting
    scored.sort(key=lambda x: x[1], reverse=True)
 
    #filtering with top-n
    top_results = scored[:top_n]
    all_scores  = scored   
 
    return top_results, matched_skills, all_scores

## **Display Helpers**

In [7]:
def print_banner():
    print("AI Recommendation Logic")
    print("Tech Stack Recommender | Content-Based Filtering")
    
 
 
def print_results(top_results, matched_skills, user_input, all_scores):
    print(f"\n User Profile ")
    print(f"  Skills entered   : {', '.join(user_input)}")
    print(f"  Skills matched   : {', '.join(matched_skills) if matched_skills else 'None (Cold Start)'}")
 
    if not top_results:
        print("\n   No matches found. Try skills like: Python, SQL,")
        print(" Machine Learning, Docker, AWS, JavaScript, Linux")
        return
 
    print(f"\n Top {len(top_results)} Recommended Career Paths ")
    for rank, (title, score, skills) in enumerate(top_results, 1):
        bar_len = int(score * 30)
        bar = "█" * bar_len + "░" * (30 - bar_len)
        print(f"\n  #{rank}  {title}")
        print(f"       Match Score : {score:.4f}  [{bar}]  {score*100:.1f}%")
        print(f"       Key Skills  : {', '.join(skills[:5])}")
 
    print(f"\n All Roles Ranked (for reference) ")
    for title, score, _ in all_scores:
        bar_len = int(score * 20)
        bar = "█" * bar_len + "░" * (20 - bar_len)
        print(f"  {score:.4f}  [{bar}]  {title}")

## **Main**

In [8]:
def main():
    print_banner()
 
    # Pre-build TF-IDF vectors once not per query
    vocab, tfidf_vectors, idf = compute_tfidf(JOB_ROLES)
 
    print(f"\n  Dataset loaded  : {len(JOB_ROLES)} job roles")
    print(f"  Vocabulary size : {len(vocab)} unique skills")
    print(f"  Algorithm       : TF-IDF + Cosine Similarity")
    print(f"\n  Example skills  : Python, SQL, Machine Learning, Docker,")
    print(f"AWS, JavaScript, React, Linux, Security\n")
 
    while True:
        
        raw = input("  Enter your skills (comma-separated, min 3): ").strip()
 
        if raw.lower() in ["exit", "quit", "bye"]:
            print("\n  Goodbye! Keep building. ")
            break
 
        # Parse input
        user_skills = [s.strip().lower() for s in raw.split(",") if s.strip()]
 
        if len(user_skills) < 3:
            print(f"  Please enter at least 3 skills for accurate matching.")
            continue
 
        # Run the 4-step pipeline
        top_results, matched, all_scores = recommend(
            user_skills, JOB_ROLES, tfidf_vectors, vocab, idf, top_n=3
        )
 
        print_results(top_results, matched, user_skills, all_scores)
 
        print("\n  Try another profile. (or type 'exit' to quit)")
 
 
if __name__ == "__main__":
    main()

AI Recommendation Logic
Tech Stack Recommender | Content-Based Filtering

  Dataset loaded  : 12 job roles
  Vocabulary size : 66 unique skills
  Algorithm       : TF-IDF + Cosine Similarity

  Example skills  : Python, SQL, Machine Learning, Docker,
AWS, JavaScript, React, Linux, Security

  Please enter at least 3 skills for accurate matching.

 User Profile 
  Skills entered   : python, aws, machine learning
  Skills matched   : python, aws, machine learning

 Top 3 Recommended Career Paths 

  #1  Machine Learning Engineer
       Match Score : 0.2229  [██████░░░░░░░░░░░░░░░░░░░░░░░░]  22.3%
       Key Skills  : python, machine learning, tensorflow, pytorch, deep learning

  #2  Data Scientist
       Match Score : 0.1981  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]  19.8%
       Key Skills  : python, machine learning, sql, statistics, data analysis

  #3  AI Research Scientist
       Match Score : 0.1929  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]  19.3%
       Key Skills  : python, deep learning, machi